## 10. The Cold Start Problem & Hybrid Solution

**The Cold Start Problem** is a notorious limitation of pure Collaborative Filtering (CF) systems:
1. **New User Problem:** A new user has rated 0 movies. The CF matrix has no latent vector for them, so we cannot predict what they will like.
2. **New Movie Problem:** A newly released movie has 0 ratings. It will never be recommended by CF, regardless of how good it is.

### How We Managed It:
- **PySpark Strategy:** In our ALS model, we explicitly set `coldStartStrategy="drop"`. This means if the model encounters a user/movie in the test set that wasn't in the training set, it drops the prediction instead of throwing a `NaN` error. This ensures our RMSE calculation remains mathematically valid.
- **The Hybrid Mitigation:** Our Content-Based Engine (TF-IDF) completely bypasses the New Movie problem. Because it relies on *Metadata* (Overview, Genres, Cast) instead of user interactions, a movie released 5 minutes ago can still be recommended to users if its metadata vector closely matches their preferences.


## 1. Environment Setup\nInstalling PySpark, Kaggle API, and downloading datasets.

In [ ]:
# Install PySpark and findspark
!pip install pyspark findspark

# Download the MovieLens 32M dataset
!wget https://files.grouplens.org/datasets/movielens/ml-32m.zip

# Unzip the downloaded file
!unzip -o ml-32m.zip\n\n# Install kaggle library if not present
!pip install -q kaggle

# --- KAGGLE SETUP ---
# You need a kaggle.json file. Run this cell to upload it directly to Colab.
import os
from google.colab import files

print("Please upload your kaggle.json file (Download it from Kaggle -> Account -> Create New API Token)")
if not os.path.exists('/root/.kaggle/kaggle.json'):
    try:
        uploaded = files.upload()
        # Move kaggle.json to the correct directory and set permissions
        !mkdir -p ~/.kaggle
        !cp kaggle.json ~/.kaggle/
        !chmod 600 ~/.kaggle/kaggle.json
        print("✅ Kaggle API key successfully configured!")
    except Exception as e:
        print(f"Upload cancelled or failed: {e}")
else:
    print("✅ Kaggle API key is already configured!")
\n\nprint("⏳ Downloading TMDB 5000 Movies Dataset...")
!kaggle datasets download -d tmdb/tmdb-movie-metadata -p tmdb --unzip

print("\n⏳ Downloading IMDb Non-Commercial Datasets...")
# Note: IMDb provides datasets publicly at https://datasets.imdbws.com/
# We'll use wget for IMDb as it's direct and doesn't require API authentication.
!mkdir -p imdb
!wget -q https://datasets.imdbws.com/title.basics.tsv.gz -O imdb/title.basics.tsv.gz
!wget -q https://datasets.imdbws.com/title.ratings.tsv.gz -O imdb/title.ratings.tsv.gz
!gunzip -f imdb/*.gz

print("\n⏳ Downloading Bollywood Dataset...")
# A popular dataset for Bollywood movies.
!kaggle datasets download -d rishidamarla/bollywood-movies-dataset -p bollywood --unzip

print("\n✅ All datasets downloaded and extracted successfully!")
\n

In [ ]:
import findspark
findspark.init()
from pyspark.sql import SparkSession

# Initialize Spark with optimized memory settings for Colab
spark = SparkSession.builder \
    .appName("BigData_Movie_Recommender") \
    .config("spark.driver.memory", "8g") \
    .config("spark.sql.shuffle.partitions", "200") \
    .getOrCreate()

print("✅ Spark Session Initialized successfully!")
\n

## 2. Dataset Loading\nLoading MovieLens 32M via PySpark and Metadata (TMDB, IMDb, Bollywood) via Pandas.

In [ ]:
# Load the datasets
movies_df = spark.read.csv("ml-32m/movies.csv", header=True, inferSchema=True)
ratings_df = spark.read.csv("ml-32m/ratings.csv", header=True, inferSchema=True)

# Display the structure of our DataFrames
print("Movies Schema:")
movies_df.printSchema()

print("\nRatings Schema:")
ratings_df.printSchema()

# Verify the row counts match the dataset documentation
print("--- Data Validation ---")
print(f"Total Movies Loaded: {movies_df.count()}")
print(f"Total Ratings Loaded: {ratings_df.count()}")

# Preview the first 5 rows of ratings
ratings_df.show(5)\n

In [ ]:
import pandas as pd
import os
import glob
import ast

print("⏳ Loading Datasets into Pandas...")

# 1. TMDB Data
tmdb_movies = pd.read_csv('tmdb/tmdb_5000_movies.csv')
tmdb_credits = pd.read_csv('tmdb/tmdb_5000_credits.csv')

# Rename 'id' to 'tmdbId' to match MovieLens links.csv
tmdb_movies.rename(columns={'id': 'tmdbId'}, inplace=True)
tmdb_credits.rename(columns={'movie_id': 'tmdbId'}, inplace=True)

# Merge TMDB movies and credits
tmdb_df = tmdb_movies.merge(tmdb_credits, on='tmdbId')

# 2. IMDb Data
print("⏳ Loading IMDb Data (This may take a minute)...")
imdb_basics_cols = ['tconst', 'titleType', 'primaryTitle', 'startYear', 'genres']
imdb_basics = pd.read_csv('imdb/title.basics.tsv', sep='\t', usecols=imdb_basics_cols, low_memory=False, na_values='\\N')

# Filter for movies only to save memory
imdb_movies = imdb_basics[imdb_basics['titleType'] == 'movie'].copy()
imdb_movies.drop(columns=['titleType'], inplace=True)

imdb_ratings = pd.read_csv('imdb/title.ratings.tsv', sep='\t')

# Format IMDb ID to match MovieLens integer format (e.g., 'tt0114709' -> 114709)
# Some tconst values might be malformed, so we coerce errors
imdb_movies['imdbId'] = pd.to_numeric(imdb_movies['tconst'].str.replace('tt', ''), errors='coerce')
imdb_ratings['imdbId'] = pd.to_numeric(imdb_ratings['tconst'].str.replace('tt', ''), errors='coerce')

# Merge IMDb movies and ratings
imdb_df = imdb_movies.merge(imdb_ratings, on='imdbId', how='left')

# 3. Bollywood Data
print("⏳ Loading Bollywood Data...")
# Find the extracted CSV from the Bollywood folder
bollywood_file = glob.glob('bollywood/*.csv')[0]
bollywood_df = pd.read_csv(bollywood_file)

print(f"✅ Loaded TMDB: {len(tmdb_df)} rows")
print(f"✅ Loaded IMDb: {len(imdb_df)} rows")
print(f"✅ Loaded Bollywood: {len(bollywood_df)} rows")
\n

## 3. Data Merging & Bollywood Classification\n**Academic Justification:** A naive classification of Bollywood movies based on missing `tmdbId` is flawed because it assumes any missing TMDB entry is a Bollywood movie. \n**Improved Approach:** We will create a robust `is_bollywood` flag by explicitly checking the original language (`hi`), production country, and membership in the explicit Bollywood dataset.\n\n*Limitations:* Some regional Indian cinema (Telugu, Tamil) might not be captured perfectly if the dataset purely labels Hindi as Bollywood, but this greatly improves accuracy over missing IDs.\n

In [ ]:
import pandas as pd

print("⏳ Merging Datasets using MovieLens links.csv as the Golden Bridge...")

# Convert MovieLens data to Pandas
movies_pd = movies_df.toPandas()
links_pd = links_df.toPandas()

# Merge MovieLens movies with their links
ml_df = movies_pd.merge(links_pd, on='movieId', how='left')
ml_df['imdbId'] = pd.to_numeric(ml_df['imdbId'], errors='coerce')
ml_df['tmdbId'] = pd.to_numeric(ml_df['tmdbId'], errors='coerce')

# 1. Merge MovieLens Links with TMDB
merged_df = ml_df.merge(tmdb_df, on='tmdbId', how='left', suffixes=('', '_tmdb'))

# 2. Merge with IMDb
merged_df = merged_df.merge(imdb_df, on='imdbId', how='left', suffixes=('', '_imdb'))

# 3. Handle Bollywood Dataset
if 'title' not in bollywood_df.columns:
    col_mapping = {c: 'title' for c in bollywood_df.columns if 'title' in c.lower() or 'movie' in c.lower()}
    if col_mapping:
        bollywood_df.rename(columns=col_mapping, inplace=True)

bolly_clean = pd.DataFrame({
    'title': bollywood_df.get('title', ''),
    'overview': bollywood_df.get('Story', bollywood_df.get('Plot', '')),
    'genres_bolly': bollywood_df.get('Genre', ''),
    'cast_bolly': bollywood_df.get('Cast', ''),
    'director_bolly': bollywood_df.get('Director', ''),
    'original_language': 'hi', # Hindi
    'is_from_bollywood_csv': True
})
bolly_clean.dropna(subset=['title'], inplace=True)

# Append Bollywood to the main dataframe
movies_metadata_df = pd.concat([merged_df, bolly_clean], ignore_index=True)
movies_metadata_df['is_from_bollywood_csv'] = movies_metadata_df['is_from_bollywood_csv'].fillna(False)

# Clean up duplicates
movies_metadata_df.drop_duplicates(subset=['title'], keep='first', inplace=True)
movies_metadata_df.reset_index(drop=True, inplace=True)

# ---> ROBUST BOLLYWOOD CLASSIFICATION <---
# A movie is Bollywood if it comes from the Bollywood CSV OR its original language in TMDB is Hindi ('hi')
movies_metadata_df['is_bollywood'] = movies_metadata_df.apply(
    lambda row: True if row['is_from_bollywood_csv'] or row['original_language'] == 'hi' else False, 
    axis=1
)

print(f"✅ Final Merged Dataset Size: {len(movies_metadata_df)} movies")
print(f"✅ Identified Bollywood Movies: {movies_metadata_df['is_bollywood'].sum()}")
\n

## 4. Exploratory Data Analysis (EDA) & Profiling\nBefore building models, we profile the dataset to justify our design decisions.\n- **Sparsity Analysis:** Confirms why Collaborative Filtering is challenging and requires Matrix Factorization (ALS).\n- **Missing Values:** Guides metadata imputation strategy for Content-Based Filtering.\n- **Distributions:** Helps understand dataset imbalance and biases.\n

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

plt.style.use('dark_background')
sns.set_palette("husl")

print("--- 1. Missing Values Analysis ---")
missing_stats = movies_metadata_df[['overview', 'genres', 'cast', 'crew', 'is_bollywood']].isnull().sum()
print(missing_stats)
print("\n*Justification:* Significant missing metadata in 'cast' and 'crew' justifies our strategy to coalesce TMDB and Bollywood data during feature engineering.")

print("\n--- 2. Sparsity Analysis ---")
total_ratings = ratings_df.count()
num_users = ratings_df.select('userId').distinct().count()
num_movies = ratings_df.select('movieId').distinct().count()
matrix_size = num_users * num_movies
sparsity = 1.0 - (total_ratings / matrix_size)
print(f"Total Users: {num_users}")
print(f"Total Movies: {num_movies}")
print(f"Total Ratings: {total_ratings}")
print(f"Matrix Sparsity: {sparsity * 100:.4f}%")
print("\n*Justification:* A sparsity of >99% is typical for recommender systems. This necessitates advanced techniques like PySpark's ALS which handles sparse matrices efficiently via latent factors.")

print("\n--- 3. Dataset Imbalance (Hollywood vs Bollywood) ---")
plt.figure(figsize=(6,4))
sns.countplot(data=movies_metadata_df, x='is_bollywood')
plt.title("Hollywood vs Bollywood Distribution")
plt.xlabel("Is Bollywood?")
plt.ylabel("Count")
plt.show()

print("\n--- 4. Top Active Users ---")
user_rating_counts = ratings_df.groupBy('userId').count().orderBy('count', ascending=False).limit(10).toPandas()
plt.figure(figsize=(10,4))
sns.barplot(data=user_rating_counts, x='userId', y='count', order=user_rating_counts['userId'])
plt.title("Top 10 Most Active Users (Long-Tail Check)")
plt.xlabel("User ID")
plt.ylabel("Number of Ratings")
plt.show()

print("\n--- 5. Rating Distribution ---")
import pyspark.sql.functions as F
import matplotlib.pyplot as plt
import pandas as pd

print("⏳ Aggregating 32M ratings... (Spark is doing the heavy lifting!)")

# Group by rating and count
rating_counts = ratings_df.groupBy("rating").count().orderBy("rating")
rating_counts.show()

# Convert ONLY the aggregated result (which is tiny) to Pandas for plotting
rating_counts_pd = rating_counts.toPandas()

# Plot the distribution
plt.figure(figsize=(10, 5))
plt.bar(rating_counts_pd['rating'].astype(str), rating_counts_pd['count'], color='#4A90E2')
plt.title('Distribution of 32 Million Ratings', fontsize=14, color='white')
plt.xlabel('Rating (Stars)', color='gray')
plt.ylabel('Count (in millions)', color='gray')
# Dark background styling to match your original notebook
plt.style.use('dark_background')

plt.title('Distribution of Ratings (Class Imbalance)')
plt.show()
print("*Justification:* Ratings are right-skewed (people rate good movies more often than bad ones). We must keep this in mind for evaluation metrics.")
\n

In [ ]:
print("⏳ Finding the most rated movies...")

# Count ratings per movie
movie_rating_counts = ratings_df.groupBy("movieId").count().withColumnRenamed("count", "num_ratings")

# Join with movies_df to get the titles, and sort descending
top_movies = movie_rating_counts.join(movies_df, "movieId", "inner") \
                                .orderBy(F.col("num_ratings").desc())

print("🎬 Top 10 Most Rated Movies in the 32M Dataset:")
top_movies.select("title", "num_ratings").show(10, truncate=False)\n\nprint("*Justification:* The popularity distribution shows a steep drop-off. We filter movies with fewer than 100 ratings to reduce noise.")\n

## 5. Preprocessing & Feature Engineering

In [ ]:
print(f"Original Ratings count: {ratings_df.count()}")

# Define minimum threshold
MIN_RATINGS = 100

# Keep only movies that have at least MIN_RATINGS
valid_movies = movie_rating_counts.filter(F.col("num_ratings") >= MIN_RATINGS).select("movieId")

# Filter the original ratings dataset
cleaned_ratings_df = ratings_df.join(valid_movies, "movieId", "inner")

print(f"Cleaned Ratings count (Movies with >100 ratings): {cleaned_ratings_df.count()}")
print("✅ Preprocessing Complete! Data is ready for Machine Learning.")\n

In [ ]:
import ast

print("⏳ Phase 2: Feature Engineering & NLP...")

# Helper functions to safely parse TMDB JSON strings
def extract_names(json_str, limit=None):
    if pd.isna(json_str):
        return ""
    try:
        # Convert string representation of list of dicts to actual list
        data = ast.literal_eval(json_str)
        if isinstance(data, list):
            names = [item['name'] for item in data if 'name' in item]
            if limit:
                names = names[:limit]
            # Remove spaces for tags (e.g., 'Tom Hanks' -> 'TomHanks') so it creates a unique tag
            return " ".join([name.replace(" ", "") for name in names])
    except:
        pass
    return ""

def extract_director(json_str):
    if pd.isna(json_str):
        return ""
    try:
        data = ast.literal_eval(json_str)
        if isinstance(data, list):
            for item in data:
                if item.get('job') == 'Director':
                    return item.get('name', '').replace(" ", "")
    except:
        pass
    return ""

# 1. Process TMDB specific columns (Cast, Crew, Keywords)
print("   - Parsing TMDB JSON columns...")
movies_metadata_df['keywords_parsed'] = movies_metadata_df['keywords'].apply(lambda x: extract_names(x))
movies_metadata_df['cast_parsed'] = movies_metadata_df['cast'].apply(lambda x: extract_names(x, limit=4))
movies_metadata_df['director_parsed'] = movies_metadata_df['crew'].apply(extract_director)

# 2. Coalesce metadata to handle missing values across Hollywood vs Bollywood
print("   - Unifying Bollywood and Hollywood metadata...")
# For Bollywood, we already have 'cast_bolly' and 'director_bolly' as strings.
def clean_bolly_names(text):
    if pd.isna(text): return ""
    # Some bollywood cast lists are comma separated. We remove spaces to form tags.
    return str(text).replace(", ", " ").replace(",", " ").replace(" ", "")

movies_metadata_df['cast_bolly_clean'] = movies_metadata_df['cast_bolly'].apply(clean_bolly_names)
movies_metadata_df['director_bolly_clean'] = movies_metadata_df['director_bolly'].apply(clean_bolly_names)

# Combine them! If TMDB is missing, use Bollywood
movies_metadata_df['final_cast'] = movies_metadata_df['cast_parsed'].fillna('') + " " + movies_metadata_df['cast_bolly_clean'].fillna('')
movies_metadata_df['final_director'] = movies_metadata_df['director_parsed'].fillna('') + " " + movies_metadata_df['director_bolly_clean'].fillna('')

# 3. Handle Genres
# MovieLens genres are pipe separated (Action|Adventure). Bollywood might be comma separated.
movies_metadata_df['genres'] = movies_metadata_df['genres'].fillna('')
movies_metadata_df['genres_bolly'] = movies_metadata_df['genres_bolly'].fillna('')
movies_metadata_df['final_genres'] = movies_metadata_df['genres'].str.replace("|", " ") + " " + movies_metadata_df['genres_bolly'].str.replace(",", " ")

# 4. Handle Overview
movies_metadata_df['final_overview'] = movies_metadata_df['overview'].fillna('')

print("✅ Extracted Cast, Director, Keywords, and Genres!")
\n

In [ ]:
import re
from nltk.stem.porter import PorterStemmer

print("⏳ Building the Combined 'tags' column...")

# Combine everything into one giant string for each movie
movies_metadata_df['tags'] = (
    movies_metadata_df['final_overview'] + " " +
    movies_metadata_df['final_genres'] + " " +
    movies_metadata_df['keywords_parsed'] + " " +
    movies_metadata_df['final_cast'] + " " +
    movies_metadata_df['final_director']
)

# Convert to lowercase
movies_metadata_df['tags'] = movies_metadata_df['tags'].str.lower()

print("⏳ Applying NLP Stemming (This takes about 30 seconds)...")
ps = PorterStemmer()

# We stem the tags to reduce words to their root (e.g., 'actions', 'action' -> 'action')
def stem_text(text):
    # Also remove special characters using regex to keep it clean
    text = re.sub(r'[^a-zA-Z\s]', '', str(text))
    y = []
    for i in text.split():
        y.append(ps.stem(i))
    return " ".join(y)

movies_metadata_df['tags'] = movies_metadata_df['tags'].apply(stem_text)

# Drop all messy intermediate columns to free up RAM!
columns_to_keep = ['movieId', 'imdbId', 'tmdbId', 'title', 'final_genres', 'final_overview', 'averageRating', 'numVotes', 'tags']
final_df = movies_metadata_df[[c for c in columns_to_keep if c in movies_metadata_df.columns]].copy()

# Generate synthetic movieIds for Bollywood movies so we can reference them later
max_ml_id = final_df['movieId'].max()
mask = final_df['movieId'].isna()
final_df.loc[mask, 'movieId'] = range(int(max_ml_id) + 1, int(max_ml_id) + 1 + mask.sum())

print(f"✅ NLP Pipeline Complete! We now have a clean dataset of shape {final_df.shape}")
final_df[['title', 'tags']].head(3)
\n

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

print("⏳ Building TF-IDF Vectorizer...")

# We limit max_features to 5000 to keep the sparse matrix very memory efficient
# stop_words='english' removes common words like 'the', 'is', 'in'
tfidf = TfidfVectorizer(max_features=5000, stop_words='english')

# Fit and transform the tags column into a Sparse Matrix
# This matrix will be of shape (89075, 5000) but takes up very little RAM!
tfidf_matrix = tfidf.fit_transform(final_df['tags'])

print(f"✅ TF-IDF Matrix Built! Shape: {tfidf_matrix.shape}")

# Create a Series for reverse title mapping (Title -> Index)
# Since MovieLens titles have years like 'Toy Story (1995)', we can create a clean version for searching
def clean_title(title):
    return str(title).lower().strip()

final_df['search_title'] = final_df['title'].apply(clean_title)
indices = pd.Series(final_df.index, index=final_df['search_title']).drop_duplicates()

print("✅ Index Mapping Created. Ready for Recommendations!")
\n

## 6. Collaborative Filtering (PySpark ALS)

In [ ]:
from pyspark.ml.recommendation import ALS
from pyspark.ml.tuning import ParamGridBuilder, TrainValidationSplit
from pyspark.ml.evaluation import RegressionEvaluator
import time

print("⏳ Splitting data into Training (80%) and Testing (20%) sets...")
(training_data, test_data) = cleaned_ratings_df.randomSplit([0.8, 0.2], seed=42)

print(f"Training rows: {training_data.count()}")
print(f"Testing rows: {test_data.count()}")

print("\n🚀 Initializing ALS Model & Hyperparameter Grid...")

# Base ALS Model
als = ALS(
    userCol="userId",
    itemCol="movieId",
    ratingCol="rating",
    coldStartStrategy="drop",
    nonnegative=True  # Academic note: Prevents negative ratings, ensuring mathematical interpretability
)

# Academic Note: We use a small ParamGrid to demonstrate tuning while avoiding Colab memory limits.
param_grid = ParamGridBuilder() \
    .addGrid(als.rank, [10, 20]) \
    .addGrid(als.regParam, [0.05, 0.1]) \
    .build()

evaluator = RegressionEvaluator(metricName="rmse", labelCol="rating", predictionCol="prediction")

# Academic Note: Using TrainValidationSplit instead of CrossValidator (k-fold) for computational feasibility on 32M rows
tvs = TrainValidationSplit(estimator=als,
                           estimatorParamMaps=param_grid,
                           evaluator=evaluator,
                           trainRatio=0.8) # 80% of training data for train, 20% for validation

print("🧠 Training and Tuning the Model... (This will take a few minutes!)")
start_time = time.time()
tvs_model = tvs.fit(training_data)
end_time = time.time()

best_model = tvs_model.bestModel
print(f"✅ Model Tuning Complete in {(end_time - start_time)/60:.2f} minutes!")
print(f"🏆 Best Rank: {best_model.rank}")
print(f"🏆 Best RegParam: {best_model._java_obj.parent().getRegParam()}")
\n

In [ ]:
print("📊 Evaluating the Best Model on Test Data...")

# Make predictions
predictions = best_model.transform(test_data)

# Set up evaluators for MAE, MSE, and RMSE
evaluator_rmse = RegressionEvaluator(metricName="rmse", labelCol="rating", predictionCol="prediction")
evaluator_mse = RegressionEvaluator(metricName="mse", labelCol="rating", predictionCol="prediction")
evaluator_mae = RegressionEvaluator(metricName="mae", labelCol="rating", predictionCol="prediction")

test_rmse = evaluator_rmse.evaluate(predictions)
test_mse = evaluator_mse.evaluate(predictions)
test_mae = evaluator_mae.evaluate(predictions)

print(f"🏆 Root Mean Squared Error (RMSE) = {test_rmse:.4f}")
print(f"🏆 Mean Squared Error (MSE)       = {test_mse:.4f}")
print(f"🏆 Mean Absolute Error (MAE)      = {test_mae:.4f}")

# Context for Presentation
if test_rmse < 0.85:
    print("🌟 Excellent performance! Predictions are highly accurate.")
elif test_rmse < 1.0:
    print("👍 Good performance! Predictions are within 1 star of actual ratings.")
else:
    print("⚠️ Model needs further tuning. Errors are greater than 1 star.")
\n

In [ ]:
import pyspark.sql.functions as F

def get_recommendations_for_user(user_id, num_recs=10):
    print(f"🔎 Fetching top {num_recs} recommendations for User {user_id}...")

    # 1. Isolate the user
    single_user = cleaned_ratings_df.filter(F.col("userId") == user_id).select("userId").distinct()

    # 2. Get the recommendations
    user_subset_recs = model.recommendForUserSubset(single_user, num_recs)

    if user_subset_recs.count() == 0:
        print("❌ User not found or not enough data.")
        return

    # 3. Format the output to be readable
    recs_exploded = user_subset_recs.select(
        "userId",
        F.explode("recommendations").alias("rec")
    ).select(
        "userId",
        F.col("rec.movieId").alias("movieId"),
        F.col("rec.rating").alias("predicted_rating")
    )

    # 4. Join with the movies dataframe to get the real titles and genres
    final_recs = recs_exploded.join(movies_df, "movieId", "inner") \
        .select("title", "genres", F.round("predicted_rating", 2).alias("predicted_rating")) \
        .orderBy(F.col("predicted_rating").desc())

    final_recs.show(truncate=False)

# Let's test it on a random user! User #123
get_recommendations_for_user(123)\n

## 7. Content-Based Filtering

In [ ]:
def recommend(movie_title, n_recommendations=10):
    movie_title_clean = str(movie_title).lower().strip()

    # Check if movie exists
    if movie_title_clean not in indices:
        # Try a partial match if exact match fails
        partial_matches = final_df[final_df['search_title'].str.contains(movie_title_clean, na=False)]
        if partial_matches.empty:
            return f"❌ Movie '{movie_title}' not found in the database."
        else:
            print(f"⚠️ Exact match not found. Using closest match: {partial_matches.iloc[0]['title']}")
            movie_title_clean = partial_matches.iloc[0]['search_title']

    # Get the index of the movie
    idx = indices[movie_title_clean]

    # If there are duplicate titles, take the first one
    if isinstance(idx, pd.Series):
        idx = idx.iloc[0]

    # Get the TF-IDF vector for this specific movie
    movie_vector = tfidf_matrix[idx]

    # Calculate cosine similarity between this movie and ALL other movies on-the-fly!
    # This is incredibly fast and saves us from building an 89k x 89k dense matrix.
    sim_scores = cosine_similarity(movie_vector, tfidf_matrix).flatten()

    # Get the indices of the top N most similar movies (ignoring the first one as it's the movie itself)
    top_indices = sim_scores.argsort()[-(n_recommendations+1):-1][::-1]

    # Return the results
    results = final_df.iloc[top_indices][['title', 'final_genres', 'averageRating']].copy()
    results['Similarity Score'] = np.round(sim_scores[top_indices], 3)

    return results

print("✅ Recommendation Engine Active!")
\n

In [ ]:
from IPython.display import display

print("🎬 Recommendations for 'Inception':")
display(recommend("Inception"))

print("\n🎬 Recommendations for '3 Idiots':")
display(recommend("3 Idiots"))
\n

### Quantitative Evaluation of Content-Based Engine
While subjective "eye-tests" (like seeing if Inception recommends other Sci-Fi movies) are useful, university-level Machine Learning projects require **Quantitative Metrics**.

We evaluate our TF-IDF engine using:
1. **Average Cosine Similarity**: How mathematically close are the Top-K recommendations to the query movie in the vector space?
2. **Precision@K (Genre Overlap)**: What percentage of the Top-K recommendations share at least one primary genre with the query movie?

*Why it matters academically*: This proves that our NLP feature engineering correctly clustered movies with similar contextual meaning, rather than just returning random results.


In [ ]:
def evaluate_cbf_engine(test_movies=['Inception (2010)', 'Toy Story (1995)', 'Fight Club (1999)'], K=10):
    print(f"📊 Evaluating TF-IDF Engine (Top-{K} Recommendations)\n")
    
    total_precision = 0
    total_avg_sim = 0
    valid_movies = 0
    
    for title in test_movies:
        movie_title_clean = str(title).lower().strip()
        if movie_title_clean not in indices:
            continue
            
        idx = indices[movie_title_clean]
        if isinstance(idx, pd.Series): idx = idx.iloc[0]
            
        # Get query movie genres
        query_genres = set(str(final_df.iloc[idx]['final_genres']).split())
        if not query_genres:
            continue
            
        # Get recommendations
        movie_vector = tfidf_matrix[idx]
        sim_scores = cosine_similarity(movie_vector, tfidf_matrix).flatten()
        top_indices = sim_scores.argsort()[-(K+1):-1][::-1]
        
        avg_sim = np.mean(sim_scores[top_indices])
        
        # Calculate Precision@K based on Genre Overlap
        hits = 0
        for i in top_indices:
            rec_genres = set(str(final_df.iloc[i]['final_genres']).split())
            # If there is an intersection in genres, we count it as a relevant hit
            if query_genres.intersection(rec_genres):
                hits += 1
                
        precision = hits / K
        
        total_precision += precision
        total_avg_sim += avg_sim
        valid_movies += 1
        
        print(f"Movie: {title}")
        print(f"  - Precision@{K} (Genre Overlap): {precision*100:.1f}%")
        print(f"  - Avg Cosine Similarity: {avg_sim:.3f}\n")
        
    print("--- Overall Engine Performance ---")
    print(f"🏆 Mean Precision@{K}: {(total_precision/valid_movies)*100:.1f}%")
    print(f"🏆 Mean Cosine Similarity: {total_avg_sim/valid_movies:.3f}")

evaluate_cbf_engine()
\n

## 8. Evaluation & Overfitting Analysis
**Academic Justification of Metrics:**
- **MAE (Mean Absolute Error):** Represents the average absolute difference between predicted and actual ratings. Highly interpretable (e.g., "off by 0.6 stars").
- **MSE (Mean Squared Error):** Penalizes larger errors more heavily.
- **RMSE (Root Mean Squared Error):** The standard metric for recommender systems. It is in the same units as the ratings (stars) but disproportionately penalizes large prediction errors, making it ideal for ensuring we don't recommend truly awful movies.

**Overfitting vs. Underfitting Analysis:**
We compare the training RMSE with the testing RMSE.
- If **Train RMSE << Test RMSE**, the model is **overfitting** (memorizing the training data).
- If both are high, it is **underfitting**.


In [ ]:
# Predict on Training Data to check for overfitting
train_predictions = best_model.transform(training_data)
train_rmse = evaluator_rmse.evaluate(train_predictions)

print("--- Overfitting Analysis ---")
print(f"Train RMSE: {train_rmse:.4f}")
print(f"Test RMSE:  {test_rmse:.4f}")

difference = test_rmse - train_rmse
print(f"Difference: {difference:.4f}")

if difference > 0.15:
    print("⚠️ Warning: Model might be overfitting. Consider increasing regParam.")
elif difference < -0.05:
    print("⚠️ Warning: Anomaly detected. Test set might not be representative.")
else:
    print("✅ The model generalizes well! The small difference indicates optimal regularization.")
\n

## 9. Bias-Variance Tradeoff Discussion

The **Bias-Variance Tradeoff** is a fundamental concept in Machine Learning, and we managed it explicitly in this PySpark ALS implementation:

1. **The `regParam` (Regularization Parameter):**
   - **Low `regParam`:** Leads to **Low Bias / High Variance**. The latent factors grow unchecked to fit the training data perfectly, resulting in **Overfitting** (poor generalization to the test set).
   - **High `regParam`:** Leads to **High Bias / Low Variance**. The model is heavily penalized, keeping weights close to zero, which can result in **Underfitting** (failing to capture complex user preferences).
   - *Our approach:* By using `TrainValidationSplit` with `[0.05, 0.1]`, we systematically searched for the optimal balance that minimizes total expected error.

2. **The `rank` Parameter (Latent Factors):**
   - Higher rank increases model complexity (Variance), while lower rank restricts it (Bias). We tuned this to avoid memorization.

3. **Non-negativity Constraint (`nonnegative=True`):**
   - Enforcing non-negative matrix factorization adds a structural bias, but it heavily reduces variance and makes the resulting user/item matrices highly interpretable in real-world recommendation scenarios.


## 10. Cold Start Problem\n*(Phase 6 Implementation will go here)*\n

## 11. Conclusion, Limitations, & Future Scope

### ✅ Conclusion
This project successfully demonstrates a full-stack Machine Learning pipeline applied to a massive Big Data problem. By utilizing **PySpark** for distributed matrix factorization, we efficiently processed the 32-Million-row MovieLens dataset. 
We introduced a **Hybrid Architecture** by integrating a high-dimensional NLP **TF-IDF engine** built on TMDB/IMDb/Bollywood metadata. This approach provides hyper-personalized recommendations via Collaborative Filtering, while overcoming the Cold-Start problem and ensuring diverse, content-aware discovery via Content-Based Filtering.

### ⚠️ Limitations
1. **Bollywood Metadata Reliance:** The NLP engine currently relies heavily on English overviews. Bollywood movies with missing or poorly translated synopses will group poorly in the vector space.
2. **Static Model:** The ALS model requires a full re-train periodically. It does not update in real-time as a user clicks around the UI (no online learning).
3. **Hardware Constraints:** While PySpark can handle cluster-scale data, tuning the `ParamGrid` exhaustively was restricted by local/Colab memory limits.

### 🚀 Future Scope
- **Deep Learning / Neural Collaborative Filtering:** Replacing ALS with a neural network architecture (e.g., embeddings combined with dense layers) to capture non-linear user-item relationships.
- **Sentiment Analysis:** Scraping IMDB reviews and adding a sentiment score to the Content-Based Engine.
- **Real-time Streaming:** Implementing Apache Kafka + Spark Streaming to update user preference vectors on the fly as they interact with the Streamlit app.
